In [ ]:
#初步查看
import pandas as pd
from rdkit import Chem

df = pd.read_csv("D:/00_vscode_project/trail_work/Lipophilicity.csv")

print(f'raw_data_rows:{len(df)}')
print()
#空值
print(df.isnull().sum())

raw_data_rows:4200

CMPD_CHEMBLID    0
exp              1
smiles           1
dtype: int64


In [ ]:
#初步清洗
from rdkit import Chem

#删空值。subset定位对应列——copy生成副本，以防在df中处理
df_clean = df.dropna(subset=['exp', 'smiles']).copy()
print(f"删除空值后的行数: {len(df_clean)}")

#非法 SMILES
def check_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

df_clean = df_clean[df_clean['smiles'].apply(check_smiles)]
print(f"过滤非法 SMILES 后的行数: {len(df_clean)}")

#查重。在df_clean里找叫drop_duplicates的函数
df_clean = df_clean.drop_duplicates(subset=['smiles'])
print(f"去重后的最终行数: {len(df_clean)}")

display(df_clean.head())

删除空值后的行数: 4198
过滤非法 SMILES 后的行数: 4196
去重后的最终行数: 3782


[22:51:11] SMILES Parse Error: syntax error while parsing: not_a_smiles
[22:51:11] SMILES Parse Error: check for mistakes around position 3:
[22:51:11] not_a_smiles
[22:51:11] ~~^
[22:51:11] SMILES Parse Error: Failed parsing SMILES 'not_a_smiles' for input: 'not_a_smiles'
[22:51:11] SMILES Parse Error: extra open parentheses while parsing: C1(CC
[22:51:11] SMILES Parse Error: check for mistakes around position 3:
[22:51:11] C1(CC
[22:51:11] ~~^
[22:51:11] SMILES Parse Error: Failed parsing SMILES 'C1(CC' for input: 'C1(CC'


,CMPD_CHEMBLID,exp,smiles
0,CHEMBL325503,1.48,CC(C)C(NC(=O)CN1C(=O)C(=CC=C1c2ccccc2)NC(=O)NC...
1,CHEMBL40,.05,CCC1(C(=O)NC(=O)NC1=O)c2ccccc2
2,CHEMBL1294,2.21,COc1ccc2nccc([C@H](O)[C@H]3C[C@@H]4CCN3C[C@@H]...
3,CHEMBL277474,.11,CN1N(C(=O)C=C1C)c2ccccc2
4,CHEMBL1299863,3.78,CCc1c(C)nc(SCC(=O)Nc2nc(cs2)c3ccc(OC)cc3)nc1O


In [ ]:
#修改数据格式
#describe看数据属性
print(df_clean['exp'].describe())
'''
    原始数据不能直接使用！
    dtype错误，如果是正常的数字，应该是float，现在它被当成字符串处理
    原因：混杂了非标准数字的数值，特殊符号，空字符等
'''
print()
#强制转换为数值型，无法转换的会变成 NaN
df_clean['exp'] = pd.to_numeric(df_clean['exp'], errors='coerce')

print(f"非纯数字导致转换失败的行数: {df_clean['exp'].isnull().sum()}")

print("捣乱的非数字数据示例：")
print(df_clean[df_clean['exp'].isnull()].head())#head()默认显示前5行

#转换失败的行删掉
df_clean = df_clean.dropna(subset=['exp']).reset_index(drop=True)
#dropna删掉空值，reset_index重新索引，drop=True不保留旧索引

#再次查看 describe，（正常则显示 mean, std, min, max）
print(df_clean['exp'].describe())

count     3782
unique     546
top        3.1
freq        55
Name: exp, dtype: object

非纯数字导致转换失败的行数: 0
捣乱的非数字数据示例：
Empty DataFrame
Columns: [CMPD_CHEMBLID, exp, smiles]
Index: []
count    3782.000000
mean        2.183258
std         1.203938
min        -1.500000
25%         1.400000
50%         2.350000
75%         3.100000
max         4.500000
Name: exp, dtype: float64


In [4]:
# 把你清洗好的 df_clean 保存为新的 csv 文件
df_clean.to_csv("D:/00_vscode_project/trail_work/s1_Lipophilicity_clean.csv", index=False)
print("数据已保存！")

数据已保存！


In [ ]:
'''
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger

#关闭日志
RDLogger.DisableLog('rdApp.*')
df = pd.read_csv("D:/00_vscode_project/trail_work/s1_Lipophilicity_clean.csv")

# 截取前 3000 个
df_part1 = df.head(3000).copy()

lfc = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()

def safe_clean_smiles(smiles):
    if not isinstance(smiles, str): return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    try:
        mol = lfc.choose(mol)
        mol = uncharger.uncharge(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

print("开始处理前 3000 行...")
df_part1['std_smiles'] = df_part1['smiles'].apply(safe_clean_smiles)
print("✅ 前 3000 行处理完成！")
'''

开始处理前 3000 行...
✅ 前 3000 行处理完成！


In [ ]:
'''
# 截取后面的数据 (3000 到末尾)
df_part2 = df.iloc[3000:].copy()
print(f"剩余待处理行数: {len(df_part2)}")

# 依然用之前的清洗函数，但是细化分批
batch_size = 100
valid_results = []

for i in range(0, len(df_part2), batch_size):
    print(f"正在处理剩余数据的第 {i} 到 {i+batch_size} 行...")
    chunk = df_part2.iloc[i:i+batch_size].copy()
    chunk['std_smiles'] = chunk['smiles'].apply(safe_clean_smiles)
    valid_results.append(chunk)

print("✅ 剩余部分处理完成！")
'''

剩余待处理行数: 782
正在处理剩余数据的第 0 到 100 行...
正在处理剩余数据的第 100 到 200 行...
正在处理剩余数据的第 200 到 300 行...
正在处理剩余数据的第 300 到 400 行...
正在处理剩余数据的第 400 到 500 行...
正在处理剩余数据的第 500 到 600 行...


: 

In [ ]:
'''
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
df = pd.read_csv("D:/00_vscode_project/trail_work/s1_Lipophilicity_clean.csv")

# 精确提取“嫌疑犯”区间：原数据的 3500 到 3600 行（不包含 3600）
suspects = df.iloc[3750:3780].copy()

lfc = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()

# 取消批量，我们一个一个枪毙（测试）
for index, row in suspects.iterrows():
    smiles = row['smiles']
    print(f"正在测试原索引 {index} 的分子...")
    print(f"SMILES: {smiles[:50]}...") # 只打印前50个字符防止刷屏
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(" -> 解析失败，跳过\n")
        continue
        
    # 这里就是案发现场
    try:
        mol = lfc.choose(mol)
        mol = uncharger.uncharge(mol)
        std_smiles = Chem.MolToSmiles(mol, canonical=True)
        print(" -> 安全存活\n")
    except Exception as e:
        print(f" -> 报错，但不至于崩溃: {e}\n")
        
***多次尝试分批处理后，将区间一次次缩小，最终定位到 3770-3780 行的 10 个分子，最后回到文件肉眼辨别***
'''

正在测试原索引 3750 的分子...
SMILES: O=S(=O)(c1ccccc1)C2(CC2)c3cc(nc(n3)c4cccc5[nH]ccc4...
 -> 安全存活

正在测试原索引 3751 的分子...
SMILES: Nc1n[nH]c(N)c1Cc2ccc3OCOc3c2...
 -> 安全存活

正在测试原索引 3752 的分子...
SMILES: C(Nc1ncnc2ccccc12)c3ccccc3...
 -> 安全存活

正在测试原索引 3753 的分子...
SMILES: CS(=O)(=O)Cc1cc(nc(n1)c2cccc3[nH]ccc23)N4CCOCC4...
 -> 安全存活

正在测试原索引 3754 的分子...
SMILES: CN[C@@H](C)C(=O)N[C@H](C(=O)N[C@H]1CCCN(CCc2ccc(cc...
 -> 安全存活

正在测试原索引 3755 的分子...
SMILES: CCN1CCC[C@H]1CNC(=O)c2cc(Br)cc(OC)c2OC...
 -> 安全存活

正在测试原索引 3756 的分子...
SMILES: Nc1nccc(n1)c2c[nH]c3ccccc23...
 -> 安全存活

正在测试原索引 3757 的分子...
SMILES: CC1=NC(C)(N=C1N)c2cccc(c2)c3cccnc3...
 -> 安全存活

正在测试原索引 3758 的分子...
SMILES: CNC(=O)c1ccc(CCC(COc2ccc(cc2)c3cccc(c3)[N+](=O)[O-...
 -> 安全存活

正在测试原索引 3759 的分子...
SMILES: Cc1ccc(cc1NC(=O)c2ccc(OCc3ccccn3)cc2)c4nccn4C...
 -> 安全存活

正在测试原索引 3760 的分子...
SMILES: CNC1=Nc2ncccc2C(=NC1c3cccs3)c4cc[nH]c4...
 -> 安全存活

正在测试原索引 3761 的分子...
SMILES: CC[C@H](CO)Nc1nc(SCc2ccccc2)nc3nc(N)sc13...
 -> 安全存活

正在测试原索引 3762 的分子...
SMI

In [ ]:
#去盐化和规范化
from rdkit import RDLogger

# 关掉 RDKit日志
RDLogger.DisableLog('rdApp.*')

print("正在读取数据...")
df = pd.read_csv("D:/00_vscode_project/trail_work/s1_Lipophilicity_clean.csv")
print(f"原始行数: {len(df)}")

#把 SMILES 长度超过 300 的直接当脏数据删掉
df = df[df['smiles'].str.len() < 300]
#把完全空白的行删掉（针对你说的最后一行空的）
df = df.dropna(subset=['smiles'])
print(f"排雷（删掉面条和空行）后行数: {len(df)}")

#安全去盐与规范化
def safe_clean_smiles(smiles):
    if not isinstance(smiles, str): 
        return None
    # 按 '.' 切分，保留最长的片段（主体分子）
    parts = smiles.split('.')
    main_part = max(parts, key=len) 
    
    # 转成 Mol 对象
    mol = Chem.MolFromSmiles(main_part)
    
    # 如果解析成功，返回规范化的 SMILES (Canonical SMILES)
    if mol is not None:
        return Chem.MolToSmiles(mol, canonical=True)
    return None

print("开始安全去盐及规范化，请稍候...")
df['std_smiles'] = df['smiles'].apply(safe_clean_smiles)

#清理空值并去重
df_final = df.dropna(subset=['std_smiles']).copy()
df_final = df_final.drop_duplicates(subset=['std_smiles'], keep='first').reset_index(drop=True)

print(f"最终有效行数: {len(df_final)}")

df_final.to_csv("D:/00_vscode_project/trail_work/s1_Lipophilicity_clean_final.csv", index=False)

正在读取数据...
原始行数: 3782
排雷（删掉面条和空行）后行数: 3780
开始安全去盐及规范化，请稍候...
🎉 最终有效行数: 3780
✅ 完美！已保存为 Lipophilicity_Final_Clean.csv
